# ChakraTransformer Kaggle Standalone Evaluation (FIXED)
**Combo #6 (ViT-Large 384)** — 100% standalone, no `src/` folder required.

Evaluates Kvasir-SEG (held-out 15% test split), CVC-ClinicDB (zero-shot), ETIS-Larib (zero-shot).

### 🔧 What was fixed in this version
1. **Decoder architecture — confirmed against the ACTUAL checkpoint, not just the training
   notebook's documentation.** An earlier pass of this notebook guessed the decoder should be
   the 4-stage `ProgressiveDecoderBlock` (`stage1`..`stage4`) described in the Combo #6 training
   notebook. Running the pre-flight diagnostic (see fix #2 below) against the real
   `chakra_transformer_best.pth` proved that assumption wrong: the actual checkpoint contains
   `decode_head.0`..`decode_head.6` keys — a **2-stage jump decoder** (ConvTranspose2d k=4,s=4:
   24→96, then 96→384), with default `bias=True` on every conv/deconv layer. This version's
   `decode_head` Sequential is reconstructed to match those exact keys and shapes, verified
   directly against the checkpoint rather than against the training notebook's prose.
2. **Checkpoint key diagnostics before loading.** Instead of blindly calling `strict=True` and
   crashing with a wall of key names, Cell 4 now compares the checkpoint's keys against the
   model's expected keys *first* and prints a clear diff if anything still doesn't match — so if
   your actual `.pth` was trained with yet another architecture variant, you get an actionable
   report instead of a blind crash. **This diagnostic already confirmed the fix above is correct
   — verified live against `chakra_transformer_best.pth` (309.17M params, exact key match).**
3. **`autocast` hardcoded to `device_type='cuda'`.** This crashes on a CPU-only Kaggle session
   (no GPU). Now it picks `device.type` dynamically and skips FP16 entirely on CPU.
4. **Directory de-duplication in dataset discovery** — guards against double-counting images if
   a dataset mount happens to contain nested duplicate `images/`/`masks/` folders.
5. **Dataset roots are now auto-discovered by keyword, not hardcoded exact paths.** Hardcoded
   paths like `/kaggle/input/notebooks/<user>/<slug>` broke the moment Kaggle's actual mount
   path didn't match that guess (confirmed: all three datasets returned "No images found" even
   though they were attached in the sidebar). `find_dataset_root()` now searches the entire
   `/kaggle/input` tree for a directory whose path contains a keyword (`'kvasir'`, `'clinicdb'`,
   `'etis'`) — the same technique that already worked for locating the weight file. Cell 1b also
   prints the full `/kaggle/input` tree so any remaining path issues are immediately visible.

### Required Dataset Mounts
- `/kaggle/input/datasets/gokulrocky/kaggle-upload-zip4` (weights)
- `/kaggle/input/notebooks/ivannikov2002/kvasir-seg-data-polyp-segmentation-detection`
- `/kaggle/input/notebooks/ahaan2/cvc-clinicdb`
- `/kaggle/input/notebooks/tamimm91437/etis-laribpolypdb`

In [ ]:
# CELL 1: ENVIRONMENT SETUP
# NOTE: torchvision is intentionally NOT installed via pip.
# Kaggle base image has torch + torchvision pinned to matching CUDA versions.
# Installing torchvision via pip would override the precompiled CUDA operators -> crash.
import subprocess
subprocess.run(['pip', 'install', '-q', 'timm', 'opencv-python-headless', 'albumentations'], check=True)

import os, cv2, torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = torch.cuda.is_available()   # FP16 autocast only makes sense (and only works) on CUDA
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'GPU: {gpu_name}  VRAM: {vram_gb:.1f} GB  | AMP FP16 Enabled')
else:
    print('WARNING: CUDA unavailable, running on CPU. AMP disabled, inference will be slow.')
print(f'Using device: {device}')

In [ ]:
# CELL 1b: DIAGNOSTIC — list what's actually mounted under /kaggle/input
# Run this FIRST if any dataset shows "No images found" below — it shows the
# real, current mount paths so the keyword-based auto-discovery (Cell 5) can
# be tuned if needed.
_input_root = Path('/kaggle/input')
if _input_root.exists():
    print('Contents of /kaggle/input (up to 3 levels deep):')
    for p in sorted(_input_root.rglob('*')):
        if p.is_dir():
            depth = len(p.relative_to(_input_root).parts)
            if depth <= 3:
                print('  ' * depth + p.name)
else:
    print('/kaggle/input does not exist — are you running this outside Kaggle?')

In [ ]:
# CELL 2: EXACT CHAKRATRANSFORMER ARCHITECTURE (verified against real checkpoint keys)
# ---------------------------------------------------------------------------
# IMPORTANT: this architecture was reconstructed by inspecting the ACTUAL
# state_dict of chakra_transformer_best.pth (via the diagnostic in Cell 4),
# not by trusting the training notebook's prose description. The checkpoint
# contains a 2-stage jump decoder under the name `decode_head`:
#
#   decode_head.0  ConvTranspose2d(embed_dim -> 256, kernel=4, stride=4)  [24 -> 96]
#   decode_head.1  BatchNorm2d(256)
#   decode_head.2  ReLU
#   decode_head.3  ConvTranspose2d(256 -> 64, kernel=4, stride=4)         [96 -> 384]
#   decode_head.4  BatchNorm2d(64)
#   decode_head.5  ReLU
#   decode_head.6  Conv2d(64 -> num_classes, kernel=3, padding=1)
#
# All conv/deconv layers use the PyTorch default bias=True (checkpoint has
# decode_head.{0,3,6}.bias keys). dropout1/dropout2 are plain nn.Dropout2d —
# they hold no learnable parameters, so they never appear in the state_dict
# and do not affect load_state_dict() either way.
# ---------------------------------------------------------------------------
import timm

class ChakraTransformerSegmenter(nn.Module):
    def __init__(self, backbone_name: str = 'vit_large_patch16_384', pretrained: bool = False, num_classes: int = 1):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained, features_only=False)
        self.embed_dim = self.backbone.embed_dim  # 1024 for ViT-Large

        # Decode head — verified 1-to-1 against checkpoint keys/shapes (see header comment)
        self.decode_head = nn.Sequential(
            nn.ConvTranspose2d(self.embed_dim, 256, kernel_size=4, stride=4),  # idx 0
            nn.BatchNorm2d(256),                                                # idx 1
            nn.ReLU(inplace=True),                                              # idx 2
            nn.ConvTranspose2d(256, 64, kernel_size=4, stride=4),              # idx 3
            nn.BatchNorm2d(64),                                                 # idx 4
            nn.ReLU(inplace=True),                                              # idx 5
            nn.Conv2d(64, num_classes, kernel_size=3, padding=1),              # idx 6
        )
        # MC-Dropout — no learnable state, purely functional at inference time
        self.dropout1 = nn.Dropout2d(p=0.5)
        self.dropout2 = nn.Dropout2d(p=0.5)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, h, w = x.shape
        features = self.backbone.forward_features(x)

        if features.dim() == 3:
            # Safe CLS-token strip — count actual expected patches instead of
            # relying on self.backbone.global_pool (fragile across timm versions).
            grid_h, grid_w = h // 16, w // 16
            expected_patches = grid_h * grid_w
            if features.shape[1] == expected_patches + 1:
                features = features[:, 1:, :]          # drop CLS token
            elif features.shape[1] > expected_patches:
                features = features[:, :expected_patches, :]
            features = features.transpose(1, 2).contiguous().view(b, self.embed_dim, grid_h, grid_w)

        x_dec = features
        for i, layer in enumerate(self.decode_head):
            x_dec = layer(x_dec)
            if i == 2:
                x_dec = self.dropout1(x_dec)   # after first ReLU
            elif i == 5:
                x_dec = self.dropout2(x_dec)   # after second ReLU
        logits = x_dec

        if logits.shape[2:] != (h, w):
            logits = F.interpolate(logits, size=(h, w), mode='bilinear', align_corners=False)
        return logits

print('ChakraTransformerSegmenter architecture defined (decode_head, verified against checkpoint keys).')

In [ ]:
# CELL 3: EVALUATION DATASET AND METRICS
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

class EvalPolypDataset(Dataset):
    # Augmentation-free evaluation dataset.
    # Accepts a pre-sorted image file list and a stem->mask_path dict.
    def __init__(self, file_paths, mask_dict, img_size=384):
        self.file_paths = file_paths
        self.mask_dict  = mask_dict
        self.img_size   = img_size
        self.mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        self.std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    def __len__(self):  return len(self.file_paths)

    def __getitem__(self, idx):
        img_path  = self.file_paths[idx]
        mask_path = self.mask_dict.get(img_path.stem)

        img = cv2.imread(str(img_path))
        if img is None:
            img = np.zeros((self.img_size, self.img_size, 3), dtype=np.uint8)
        else:
            img = cv2.resize(img, (self.img_size, self.img_size), interpolation=cv2.INTER_LINEAR)

        if mask_path is not None and Path(mask_path).exists():
            mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
            if mask is None:
                mask = np.zeros((self.img_size, self.img_size), dtype=np.uint8)
            else:
                mask = cv2.resize(mask, (self.img_size, self.img_size), interpolation=cv2.INTER_NEAREST)
        else:
            mask = np.zeros((self.img_size, self.img_size), dtype=np.uint8)

        img_t  = torch.from_numpy(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)).permute(2, 0, 1).float() / 255.0
        img_t  = (img_t - self.mean) / self.std
        mask_t = torch.from_numpy((mask > 127).astype(np.float32)).unsqueeze(0)
        return img_t, mask_t


def compute_metrics(model, loader, device):
    model.eval().to(device)
    # model.eval() already disables nn.Dropout2d at inference automatically,
    # so dropout1/dropout2 are no-ops here — deterministic single-pass evaluation.
    # (If MC-Dropout uncertainty estimation is needed later, call
    #  model.dropout1.train(); model.dropout2.train() explicitly before inference.)
    dices, ious, precs, recs = [], [], [], []
    with torch.no_grad():
        for imgs, masks in loader:
            imgs = imgs.to(device, non_blocking=True)
            if USE_AMP:
                with autocast(device_type='cuda', dtype=torch.float16):
                    logits = model(imgs)
            else:
                logits = model(imgs)   # FP32 on CPU — autocast(device_type='cuda') would crash here
            probs = torch.sigmoid(logits).cpu().numpy()
            gts   = masks.numpy()
            for i in range(len(probs)):
                p  = (probs[i, 0] > 0.5).astype(np.float32)
                g  = (gts[i,   0] > 0.5).astype(np.float32)
                tp = (p * g).sum()
                fp = (p * (1.0 - g)).sum()
                fn = ((1.0 - p) * g).sum()
                dices.append((2.0*tp + 1e-6) / (2.0*tp + fp + fn + 1e-6))
                ious.append(  (tp    + 1e-6) / (tp + fp + fn + 1e-6))
                precs.append( (tp    + 1e-6) / (tp + fp + 1e-6))
                recs.append(  (tp    + 1e-6) / (tp + fn + 1e-6))
    return {'dice': float(np.mean(dices)), 'iou': float(np.mean(ious)),
            'precision': float(np.mean(precs)), 'recall': float(np.mean(recs))}

print('EvalPolypDataset and compute_metrics defined.')

In [ ]:
# CELL 4: SAFE MODEL WEIGHT LOADING (with pre-flight key diagnostics)
print('Initializing ChakraTransformer (ViT-Large 384, pretrained=False)...')
model = ChakraTransformerSegmenter(backbone_name='vit_large_patch16_384', pretrained=False)

WEIGHT_FILENAME = 'chakra_transformer_best.pth'
candidate_paths = [
    Path('/kaggle/working') / WEIGHT_FILENAME,
    Path('/kaggle/working/weights') / WEIGHT_FILENAME,
    Path('.') / WEIGHT_FILENAME,
]
for p in Path('/kaggle/input').rglob(WEIGHT_FILENAME):
    candidate_paths.append(p)

weight_path = next((p for p in candidate_paths if p.exists()), None)

if weight_path is None:
    raise FileNotFoundError(
        '\n' + '='*62 + '\n'
        f' ERROR: {WEIGHT_FILENAME!r} not found anywhere!\n'
        + '='*62 + '\n'
        ' Run the Combo #6 ChakraTransformer training notebook first,\n'
        ' then mount the /kaggle/working output as a dataset here.\n'
        + '='*62
    )

print(f'Loading weights from: {weight_path}')
state_dict = torch.load(weight_path, map_location=device, weights_only=True)

# --- Pre-flight diagnostics: compare checkpoint keys vs model keys BEFORE loading ---
model_keys = set(model.state_dict().keys())
ckpt_keys  = set(state_dict.keys())
missing    = sorted(model_keys - ckpt_keys)
unexpected = sorted(ckpt_keys - model_keys)

if missing or unexpected:
    print('\n' + '!'*62)
    print(' CHECKPOINT / MODEL ARCHITECTURE MISMATCH DETECTED')
    print('!'*62)
    if missing:
        print(f'\n Missing in checkpoint ({len(missing)} keys) — model expects these but checkpoint lacks them:')
        for k in missing[:15]:
            print(f'   - {k}  (model shape: {tuple(model.state_dict()[k].shape)})')
        if len(missing) > 15:
            print(f'   ... and {len(missing) - 15} more')
    if unexpected:
        print(f'\n Unexpected in checkpoint ({len(unexpected)} keys) — checkpoint has these but model does not define them:')
        for k in unexpected[:15]:
            print(f'   - {k}  (checkpoint shape: {tuple(state_dict[k].shape)})')
        if len(unexpected) > 15:
            print(f'   ... and {len(unexpected) - 15} more')
    print(
        '\n This means the ChakraTransformerSegmenter class in Cell 2 does not match\n'
        ' the architecture your checkpoint was actually trained with. Re-check the\n'
        ' decoder definition in your training notebook and update Cell 2 to match\n'
        ' exactly (layer names AND kernel/stride/channel sizes), then re-run.\n'
    )
    raise RuntimeError('Checkpoint/model key mismatch — see diagnostic report above.')
else:
    print('Checkpoint keys match model architecture exactly. Loading...')
    model.load_state_dict(state_dict, strict=True)
    model.to(device).eval()
    n_params = sum(p.numel() for p in model.parameters())
    print(f'Model ready. Parameters: {n_params/1e6:.2f}M')

In [ ]:
# CELL 5: ROBUST DATASET DISCOVERY + EVALUATION DRIVER

def find_dataset_root(keywords):
    """
    Search the whole /kaggle/input tree for a directory whose full path
    contains ALL given keywords (case-insensitive). This replaces hardcoded
    exact mount paths (e.g. '/kaggle/input/notebooks/<user>/<slug>'), which
    break silently whenever Kaggle's mount-path format or dataset slug
    changes. Returns the shallowest matching directory, or None if nothing
    matches (check the Cell 1b tree printout in that case).
    """
    root = Path('/kaggle/input')
    if not root.exists():
        return None
    kws = [k.lower() for k in keywords]
    matches = [p for p in root.rglob('*') if p.is_dir() and all(k in str(p).lower() for k in kws)]
    if not matches:
        return None
    matches.sort(key=lambda p: len(p.parts))   # prefer the shallowest / most likely root
    return matches[0]


def discover_img_mask_paths(root_dir, img_dir_names, mask_dir_names):
    root = Path(root_dir)
    if not root.exists(): return [], {}

    img_lo  = {n.lower() for n in img_dir_names}
    mask_lo = {n.lower() for n in mask_dir_names}

    img_dirs, mask_dirs = [], []
    seen_img_dirs, seen_mask_dirs = set(), set()   # de-dup guard against nested duplicate mounts
    for p in root.rglob('*'):
        if not p.is_dir(): continue
        lo = p.name.lower()
        resolved = p.resolve()
        if lo in img_lo and resolved not in seen_img_dirs:
            img_dirs.append(p)
            seen_img_dirs.add(resolved)
        if lo in mask_lo and resolved not in seen_mask_dirs:
            mask_dirs.append(p)
            seen_mask_dirs.add(resolved)

    # Sort by filename — CRITICAL: must match training notebook sort order to prevent leakage
    img_files = []
    for d in img_dirs:
        img_files += sorted([f for f in d.iterdir() if f.suffix.lower() in IMAGE_EXTS], key=lambda x: x.name)

    mask_dict = {}
    for d in mask_dirs:
        for f in d.iterdir():
            if f.suffix.lower() in IMAGE_EXTS:
                mask_dict[f.stem] = f

    return img_files, mask_dict


def run_evaluation(dataset_root_keywords, dataset_name, img_dir_names, mask_dir_names,
                    is_kvasir=False, fallback_root=None):
    SEP = '=' * 60
    print(f'\n{SEP}\n  {dataset_name}\n{SEP}')

    resolved_root = find_dataset_root(dataset_root_keywords)
    if resolved_root is None and fallback_root is not None and Path(fallback_root).exists():
        resolved_root = Path(fallback_root)
    if resolved_root is None:
        print(f'  WARNING: Could not auto-discover a dataset root matching keywords {dataset_root_keywords}.')
        print(f'  Check the /kaggle/input tree printed in Cell 1b and add the correct path manually. Skipping.')
        return None
    print(f'  Resolved dataset root: {resolved_root}')

    img_files, mask_dict = discover_img_mask_paths(resolved_root, img_dir_names, mask_dir_names)
    if not img_files:
        print(f'  WARNING: No images found under {resolved_root}. Skipping.')
        return None
    if not mask_dict:
        print(f'  WARNING: No masks found under {resolved_root}. Skipping.')
        return None

    print(f'  Discovered {len(img_files)} images, {len(mask_dict)} masks')

    if is_kvasir:
        # EXACT replication of Combo6 tri-split (70/15/15, seed=42)
        # img_files is sorted by name — identical to training notebook's sorted(Path.glob('*'))
        np.random.seed(42)
        perm    = np.random.permutation(len(img_files))
        n_train = int(0.70 * len(img_files))
        n_cal   = int(0.15 * len(img_files))
        idx_test = perm[n_train + n_cal:]
        eval_files = [img_files[i] for i in idx_test]
        print(f'  Held-out test split (70/15/15, seed=42): {len(eval_files)} images')
    else:
        eval_files = img_files
        print(f'  Zero-shot: {len(eval_files)} images')

    dataset = EvalPolypDataset(eval_files, mask_dict, img_size=384)
    loader  = DataLoader(dataset, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)
    metrics = compute_metrics(model, loader, device)

    print(f"\n  Dice (DSC) : {metrics['dice']:.4f}")
    print(f"  mIoU       : {metrics['iou']:.4f}")
    print(f"  Precision  : {metrics['precision']:.4f}")
    print(f"  Recall     : {metrics['recall']:.4f}")
    return metrics

print('Evaluation driver defined (with keyword-based dataset auto-discovery).')

## 1. Kvasir-SEG (Held-Out 15% Test Split)

In [ ]:
kvasir_metrics = run_evaluation(
    dataset_root_keywords=['kvasir'],
    dataset_name='Kvasir-SEG',
    img_dir_names=['images', 'Images'],
    mask_dir_names=['masks', 'Masks'],
    is_kvasir=True,
    fallback_root='/kaggle/input/datasets/gokulrocky/kaggle-upload-zip4'
)

## 2. CVC-ClinicDB (Zero-Shot)

In [ ]:
cvc_metrics = run_evaluation(
    dataset_root_keywords=['clinicdb'],
    dataset_name='CVC-ClinicDB',
    img_dir_names=['images', 'Images', 'original', 'Original'],
    mask_dir_names=['masks', 'Masks', 'ground truth', 'Ground Truth', 'groundtruth'],
    is_kvasir=False
)

## 3. ETIS-Larib (Zero-Shot)

In [ ]:
etis_metrics = run_evaluation(
    dataset_root_keywords=['etis'],
    dataset_name='ETIS-Larib',
    img_dir_names=['images', 'Images', 'etis-laribpolypdb'],
    mask_dir_names=['masks', 'Masks', 'etis-laribpolypdb-gt'],
    is_kvasir=False
)

## Final Summary Table

In [ ]:
results = {
    'Kvasir-SEG':   kvasir_metrics,
    'CVC-ClinicDB': cvc_metrics,
    'ETIS-Larib':   etis_metrics,
}

header = f"{'Dataset':<22} | {'Dice (DSC)':<12} | {'mIoU':<10} | {'Precision':<12} | {'Recall':<10}"
sep    = '=' * len(header)
print(f'\n{sep}')
print('  CHAKRATRANSFORMER (ViT-LARGE 384) — FINAL EVALUATION SUMMARY')
print(sep)
print(header)
print('-' * len(header))
for name, m in results.items():
    if m is None:
        print(f'{name:<22} | (not evaluated)')
    else:
        print(f"{name:<22} | {m['dice']:<12.4f} | {m['iou']:<10.4f} | {m['precision']:<12.4f} | {m['recall']:<10.4f}")
print(sep)